# Anemia Eye / Conjunctiva Training Only

Notebook này đã được chỉnh để **chỉ lấy phần ảnh mắt/kết mạc `Conjuctiva`** từ dataset Kaggle `clean-augmented-anemia-dataset`.

Không dùng các phần:
- `Finger_Nails`
- `Palm`

Model mặc định: **MobileNetV2 transfer learning**. Có thể đổi sang `resnet50` hoặc `densenet201` ở biến `MODEL_NAME`.


In [ ]:
from pathlib import Path
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)


In [ ]:
INPUT_ROOT = Path("/kaggle/input/datasets")

print("Cấu trúc /kaggle/input hiện tại:")
for root, dirs, files in os.walk(INPUT_ROOT):
    level = root.replace(str(INPUT_ROOT), "").count(os.sep)
    if level <= 5:
        print("  " * level + Path(root).name)


In [ ]:
def normalize_name(name: str) -> str:
    return name.lower().replace("_", "").replace("-", "").replace(" ", "")

def count_images(folder: Path) -> int:
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sum(1 for p in folder.rglob("*") if p.suffix.lower() in img_exts)

keywords = ["conjuctiva", "conjunctiva", "eye", "eyes", "ocular"]
eye_candidates = []

for p in INPUT_ROOT.rglob("*"):
    if p.is_dir():
        name = normalize_name(p.name)
        if any(k in name for k in keywords):
            n_images = count_images(p)
            if n_images > 0:
                eye_candidates.append((n_images, p))

eye_candidates = sorted(eye_candidates, reverse=True, key=lambda x: x[0])

print("Các folder nghi là phần mắt tìm được:")
for i, (n, p) in enumerate(eye_candidates):
    print(f"{i}: {n} ảnh | {p}")

if len(eye_candidates) == 0:
    raise FileNotFoundError(
        "Không tìm thấy folder Conjuctiva/Conjunctiva/Eye trong /kaggle/input. "
        "Hãy Add Input dataset: t2obd1a1253kmit/clean-augmented-anemia-dataset rồi chạy lại."
    )

EYE_DIR = eye_candidates[0][1]
print("\nSẽ dùng folder mắt:", EYE_DIR)


In [ ]:
# =========================
# 4. COPY RIÊNG PHẦN MẮT SANG /kaggle/working
# =========================
# Mục đích: working folder chỉ còn dataset mắt, không lẫn Finger_Nails/Palm.

WORK_DATASET_DIR = Path("/kaggle/working/anemia_eye_conjuctiva_only")

if WORK_DATASET_DIR.exists():
    shutil.rmtree(WORK_DATASET_DIR)

shutil.copytree(EYE_DIR, WORK_DATASET_DIR)

print("Đã copy riêng phần mắt vào:", WORK_DATASET_DIR)

print("\nCấu trúc dataset mắt:")
for root, dirs, files in os.walk(WORK_DATASET_DIR):
    level = root.replace(str(WORK_DATASET_DIR), "").count(os.sep)
    if level <= 3:
        img_count = count_images(Path(root))
        print("  " * level + f"{Path(root).name} | {img_count} ảnh")


In [ ]:
# =========================
# 5. TÌM TRAIN / VALIDATION / TEST
# =========================
def find_split_dir(base_dir: Path, possible_names):
    possible_names = {normalize_name(x) for x in possible_names}
    for p in base_dir.iterdir():
        if p.is_dir() and normalize_name(p.name) in possible_names:
            return p
    return None

train_dir = find_split_dir(WORK_DATASET_DIR, ["Training", "Train"])
valid_dir = find_split_dir(WORK_DATASET_DIR, ["Validation", "Valid", "Val"])
test_dir  = find_split_dir(WORK_DATASET_DIR, ["Testing", "Test"])

print("train_dir:", train_dir)
print("valid_dir:", valid_dir)
print("test_dir :", test_dir)

if train_dir is None:
    raise FileNotFoundError("Không tìm thấy thư mục Training/Train trong phần mắt.")
if valid_dir is None:
    raise FileNotFoundError("Không tìm thấy thư mục Validation/Valid/Val trong phần mắt.")

print("\nSố ảnh từng split:")
for name, split_dir in [("Training", train_dir), ("Validation", valid_dir), ("Testing", test_dir)]:
    if split_dir is not None:
        print(f"\n{name}: {count_images(split_dir)} ảnh")
        for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            print(f"  {class_dir.name}: {count_images(class_dir)} ảnh")


In [ ]:
# =========================
# 6. CONFIG TRAINING
# =========================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50

# Chọn 1 model:
#   "mobilenetv2"  : nhẹ, train nhanh, phù hợp làm demo/web/app
#   "resnet50"    : nặng hơn
#   "densenet201" : nặng nhất, có thể tốt nhưng train lâu
MODEL_NAME = "mobilenetv2"

OUTPUT_MODEL_PATH = f"/kaggle/working/best_anemia_eye_{MODEL_NAME}.keras"
FINAL_MODEL_PATH = f"/kaggle/working/final_anemia_eye_{MODEL_NAME}.keras"
LABEL_MAP_PATH = "/kaggle/working/anemia_eye_label_map.json"

print("MODEL_NAME:", MODEL_NAME)
print("OUTPUT_MODEL_PATH:", OUTPUT_MODEL_PATH)


In [ ]:
# =========================
# 7. CHỌN PREPROCESS + BASE MODEL
# =========================
def get_base_model_and_preprocess(model_name: str):
    model_name = model_name.lower()

    if model_name == "mobilenetv2":
        from tensorflow.keras.applications import MobileNetV2
        from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
        base_model = MobileNetV2(
            weights="imagenet",
            include_top=False,
            input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
        )
        return base_model, preprocess_input

    if model_name == "resnet50":
        from tensorflow.keras.applications import ResNet50
        from tensorflow.keras.applications.resnet50 import preprocess_input
        base_model = ResNet50(
            weights="imagenet",
            include_top=False,
            input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
        )
        return base_model, preprocess_input

    if model_name == "densenet201":
        from tensorflow.keras.applications import DenseNet201
        from tensorflow.keras.applications.densenet import preprocess_input
        base_model = DenseNet201(
            weights="imagenet",
            include_top=False,
            input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
        )
        return base_model, preprocess_input

    raise ValueError("MODEL_NAME chỉ nhận: mobilenetv2, resnet50, densenet201")

base_model, preprocess_input = get_base_model_and_preprocess(MODEL_NAME)
print("Loaded base model:", MODEL_NAME)


In [ ]:
# =========================
# 8. IMAGE GENERATORS
# =========================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True
)

valid_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True
)

valid_generator = valid_test_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_generator = None
if test_dir is not None:
    test_generator = valid_test_datagen.flow_from_directory(
        test_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="binary",
        shuffle=False
    )

print("\nClass indices:", train_generator.class_indices)

with open(LABEL_MAP_PATH, "w", encoding="utf-8") as f:
    json.dump(train_generator.class_indices, f, ensure_ascii=False, indent=2)

print("Đã lưu label map:", LABEL_MAP_PATH)


In [ ]:
# =========================
# 9. BUILD MODEL
# =========================
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()


In [ ]:
# =========================
# 10. TRAINING
# =========================
callbacks = [
    ModelCheckpoint(
        OUTPUT_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

model.save(FINAL_MODEL_PATH)

print("Best model:", OUTPUT_MODEL_PATH)
print("Final model:", FINAL_MODEL_PATH)


In [ ]:
# =========================
# 11. PLOT TRAINING RESULT
# =========================
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# =========================
# 12. EVALUATION FUNCTION
# =========================
best_model = load_model(OUTPUT_MODEL_PATH)

def evaluate_generator(generator, title="Evaluation"):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    result = best_model.evaluate(generator, verbose=1)
    print("Metric names:", best_model.metrics_names)
    print("Result:", result)

    y_prob = best_model.predict(generator)
    y_pred = (y_prob >= 0.5).astype(int).flatten()
    y_true = generator.classes

    class_names = list(generator.class_indices.keys())

    print("\nClassification report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    ))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(values_format="d")
    plt.title(title + " - Confusion Matrix")
    plt.show()

evaluate_generator(valid_generator, "Validation")

if test_generator is not None:
    evaluate_generator(test_generator, "Testing")
else:
    print("Không có thư mục Testing/Test, chỉ đánh giá bằng Validation.")


In [ ]:
# =========================
# 13. TEST 1 ẢNH BẤT KỲ
# =========================
from tensorflow.keras.preprocessing.image import load_img, img_to_array

class_indices = train_generator.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

def predict_one_image(image_path: str):
    img = load_img(image_path, target_size=IMG_SIZE)
    arr = img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr)

    prob = float(best_model.predict(arr, verbose=0)[0][0])
    pred_idx = 1 if prob >= 0.5 else 0
    pred_class = idx_to_class[pred_idx]

    print("Image:", image_path)
    print("Probability class index 1:", prob)
    print("Prediction:", pred_class)

    plt.imshow(load_img(image_path))
    plt.axis("off")
    plt.title(f"{pred_class} | prob={prob:.4f}")
    plt.show()

    return pred_class, prob

# Ví dụ lấy tự động 1 ảnh từ validation:
sample_image = next(valid_dir.rglob("*.png"), None)
if sample_image is None:
    sample_image = next(valid_dir.rglob("*.jpg"), None)

if sample_image is not None:
    predict_one_image(str(sample_image))
else:
    print("Không tìm thấy ảnh mẫu trong validation.")


In [ ]:
# =========================
# 14. NÉN MODEL ĐỂ DOWNLOAD
# =========================
import zipfile

zip_path = f"/kaggle/working/anemia_eye_{MODEL_NAME}_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in [OUTPUT_MODEL_PATH, FINAL_MODEL_PATH, LABEL_MAP_PATH]:
        if Path(file_path).exists():
            zf.write(file_path, arcname=Path(file_path).name)

print("File zip để download:", zip_path)
